In [21]:
import os
import zipfile
import shutil
from google.colab import drive

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/Origin_Challenge_Submission'
os.makedirs(f'{DRIVE_BASE}/task_1_landmark/Model_Weights', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/task_2_segmentation/Model_Weights', exist_ok=True)

# Create clean directories
!rm -rf /content/dataset
os.makedirs('/content/dataset/images', exist_ok=True)
os.makedirs('/content/dataset/masks', exist_ok=True)

def unzip_and_flatten(zip_path, target_dir):
    print(f"Checking for zip at: {zip_path}")
    if os.path.exists(zip_path):
        temp_dir = "/content/temp_extract"
        if os.path.exists(temp_dir): shutil.rmtree(temp_dir)

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(temp_dir)

        count = 0
        for root, dirs, files in os.walk(temp_dir):
            for file in files:
                if file.lower().endswith('.png'):
                    shutil.move(os.path.join(root, file), os.path.join(target_dir, file))
                    count += 1
        shutil.rmtree(temp_dir)
        print(f"Extracted and moved {count} files to {target_dir}")
    else:
        print(f"Warning: Zip file not found at {zip_path}")

# Updated paths based on actual file names found in Drive root
img_zip = "/content/images-20260823T105631Z-1-001.zip"
# Correcting the mask zip name based on kernel file list
mask_zip = "/content/drive/MyDrive/task_2_segmentation-20260823T110214Z-1-001.zip"

print("Processing images...")
unzip_and_flatten(img_zip, "/content/dataset/images")

print("Processing masks...")
unzip_and_flatten(mask_zip, "/content/dataset/masks")

print("\nVerifying final counts:")
print(f"Images found: {len(os.listdir('/content/dataset/images'))}")
print(f"Masks found: {len(os.listdir('/content/dataset/masks'))}")

print("Setup complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing images...
Checking for zip at: /content/images-20260823T105631Z-1-001.zip
Extracted and moved 622 files to /content/dataset/images
Processing masks...
Checking for zip at: /content/drive/MyDrive/task_2_segmentation-20260823T110214Z-1-001.zip
Extracted and moved 642 files to /content/dataset/masks

Verifying final counts:
Images found: 622
Masks found: 641
Setup complete!


loading the dataset

In [13]:
import os
import cv2
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split

class FetalUltrasoundDataset(Dataset):
    def __init__(self, csv_file, img_dir, mask_dir, task='A', img_size=256):
        raw_data = pd.read_csv(csv_file)
        self.task = task
        self.img_size = img_size

        print(f"Scanning directories and cleaning data for Task {task}...")
        self.img_paths = self._build_file_map(img_dir)
        self.mask_paths = self._build_file_map(mask_dir)

        valid_data = []
        for idx, row in raw_data.iterrows():
            img_name = row['image_name']

            # Skip if original image is missing
            if img_name not in self.img_paths:
                continue

            if self.task == 'B':
                # Check for mask, handling potential typos in the company's dataset
                mask_name = img_name.replace('.png', '_Annotation.png')
                alt_mask_name = img_name.replace('.png', '_annotation.png')

                if mask_name in self.mask_paths:
                    row['mask_path'] = self.mask_paths[mask_name]
                    row['img_path'] = self.img_paths[img_name]
                    valid_data.append(row)
                elif alt_mask_name in self.mask_paths:
                    row['mask_path'] = self.mask_paths[alt_mask_name]
                    row['img_path'] = self.img_paths[img_name]
                    valid_data.append(row)
            else:
                # Task A only needs the image
                row['img_path'] = self.img_paths[img_name]
                valid_data.append(row)

        self.data = pd.DataFrame(valid_data).reset_index(drop=True)
        print(f"Found {len(self.data)} valid, complete samples out of {len(raw_data)} total rows.")

    def _build_file_map(self, directory):
        # Creates a fast lookup dictionary of all files inside the dataset folders
        file_map = {}
        for root, _, files in os.walk(directory):
            for f in files:
                file_map[f] = os.path.join(root, f)
        return file_map

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = cv2.imread(row['img_path'], cv2.IMREAD_GRAYSCALE)
        orig_h, orig_w = image.shape

        image = cv2.resize(image, (self.img_size, self.img_size)) / 255.0
        image_tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0)

        if self.task == 'A':
            # Safely extract exactly the 8 coordinate columns
            coords = ['ofd_1_x', 'ofd_1_y', 'ofd_2_x', 'ofd_2_y', 'bpd_1_x', 'bpd_1_y', 'bpd_2_x', 'bpd_2_y']
            landmarks = row[coords].values.astype('float32')

            scaled_landmarks = np.zeros_like(landmarks)
            scaled_landmarks[0::2] = landmarks[0::2] * (self.img_size / orig_w)
            scaled_landmarks[1::2] = landmarks[1::2] * (self.img_size / orig_h)
            return image_tensor, torch.tensor(scaled_landmarks, dtype=torch.float32)

        elif self.task == 'B':
            mask = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (self.img_size, self.img_size))
            mask = (mask > 127).astype(np.float32)
            return image_tensor, torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

Train Task A (Landmarks)

In [14]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

class LandmarkNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet18(pretrained=True)
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.model.fc = nn.Linear(self.model.fc.in_features, 8)
    def forward(self, x): return self.model(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load data from local Colab storage
dataset_A = FetalUltrasoundDataset('/content/role_challenge_dataset_ground_truth.csv', '/content/dataset/images', '/content/dataset/masks', task='A')
train_len = int(0.8 * len(dataset_A))
train_A, val_A = random_split(dataset_A, [train_len, len(dataset_A) - train_len])
train_loader_A = DataLoader(train_A, batch_size=16, shuffle=True)

model_A = LandmarkNet().to(device)
optimizer_A = optim.Adam(model_A.parameters(), lr=1e-3)
criterion_A = nn.MSELoss()

print("Training Task A...")
for epoch in range(15):
    model_A.train()
    running_loss = 0.0
    for imgs, marks in train_loader_A:
        imgs, marks = imgs.to(device), marks.to(device)
        optimizer_A.zero_grad()
        loss = criterion_A(model_A(imgs), marks)
        loss.backward()
        optimizer_A.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader_A):.2f}")

# Save directly to your mounted Google Drive
save_path_A = f'{DRIVE_BASE}/task_1_landmark/Model_Weights/hypothesis_final_saved_model.pth'
torch.save(model_A.state_dict(), save_path_A)
print(f"Task A Weights successfully saved to Google Drive: {save_path_A}")

Scanning directories and cleaning data for Task A...
Found 622 valid, complete samples out of 622 total rows.


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Training Task A...
Epoch 1, Loss: 17700.16
Epoch 2, Loss: 12717.34
Epoch 3, Loss: 8373.14
Epoch 4, Loss: 5202.63
Epoch 5, Loss: 3122.19
Epoch 6, Loss: 2000.07
Epoch 7, Loss: 1465.26
Epoch 8, Loss: 1367.98
Epoch 9, Loss: 1057.25
Epoch 10, Loss: 1099.92
Epoch 11, Loss: 1071.16
Epoch 12, Loss: 928.30
Epoch 13, Loss: 976.63
Epoch 14, Loss: 930.44
Epoch 15, Loss: 973.89
Task A Weights successfully saved to Google Drive: /content/drive/MyDrive/Origin_Challenge_Submission/task_1_landmark/Model_Weights/hypothesis_final_saved_model.pth


Train Task B (Segmentation)

In [22]:

!pip install segmentation-models-pytorch -q
import segmentation_models_pytorch as smp

# Helper to debug directory contents
def debug_paths(path):
    if os.path.exists(path):
        files = []
        for root, _, f_list in os.walk(path):
            for f in f_list:
                files.append(f)
        print(f"Path {path} contains {len(files)} files. First 5: {files[:5]}")
    else:
        print(f"Path {path} does not exist!")

debug_paths('/content/dataset/images')
debug_paths('/content/dataset/masks')

#Data Setup
dataset_B = FetalUltrasoundDataset('/content/role_challenge_dataset_ground_truth.csv', '/content/dataset/images', '/content/dataset/masks', task='B')

if len(dataset_B) == 0:
    print("ERROR: Dataset is still empty. Please check if filenames in CSV match the file list above.")
else:
    train_len = int(0.8 * len(dataset_B))
    val_len = len(dataset_B) - train_len
    train_B, val_B = random_split(dataset_B, [train_len, val_len])
    train_loader_B = DataLoader(train_B, batch_size=8, shuffle=True)

    #U-Net Model & Loss
    model_B = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=1, classes=1).to(device)
    optimizer_B = optim.Adam(model_B.parameters(), lr=1e-3)
    criterion_B = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)

    print("Training Task B (Segmentation)...")
    for epoch in range(15):
        model_B.train()
        running_loss = 0.0
        for imgs, masks in train_loader_B:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer_B.zero_grad()
            loss = criterion_B(model_B(imgs), masks)
            loss.backward()
            optimizer_B.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader_B):.4f}")

    save_path_B = f'{DRIVE_BASE}/task_2_segmentation/Model_Weights/hypothesis_final_saved_model.pth'
    torch.save(model_B.state_dict(), save_path_B)
    print(f"Saved Task B Weights to {save_path_B}")

Path /content/dataset/images contains 622 files. First 5: ['260_2HC.png', '329_HC.png', '093_HC.png', '076_HC.png', '199_HC.png']
Path /content/dataset/masks contains 641 files. First 5: ['260_2HC.png', '093_HC.png', '176_HC_Annotation.png', '189_HC_Annotation.png', '181_HC_Annotation.png']
Scanning directories and cleaning data for Task B...
Found 308 valid, complete samples out of 622 total rows.
Training Task B (Segmentation)...
Epoch 1, Loss: 0.9363
Epoch 2, Loss: 0.8452
Epoch 3, Loss: 0.7959
Epoch 4, Loss: 0.7840
Epoch 5, Loss: 0.7716
Epoch 6, Loss: 0.7666
Epoch 7, Loss: 0.7561
Epoch 8, Loss: 0.7499
Epoch 9, Loss: 0.7493
Epoch 10, Loss: 0.7364
Epoch 11, Loss: 0.7228
Epoch 12, Loss: 0.7331
Epoch 13, Loss: 0.7069
Epoch 14, Loss: 0.7020
Epoch 15, Loss: 0.7035
Saved Task B Weights to /content/drive/MyDrive/Origin_Challenge_Submission/task_2_segmentation/Model_Weights/hypothesis_final_saved_model.pth
